In [1]:
import time
import threading
import numpy as np
import torch
import torch.nn.functional as F
import cv2
import ipywidgets as widgets
from IPython.display import display

from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

from jetracer.nvidia_racecar import NvidiaRacecar
from torch2trt import TRTModule
from dt_apriltags import Detector

from utils import preprocess


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


In [2]:
# Road following model
road_model = TRTModule()
road_model.load_state_dict(torch.load('road_following_model_2_trt.pth'))
road_model.eval()

# Collision avoidance model
avoid_model = TRTModule()
avoid_model.load_state_dict(torch.load('best_model_resnet20_trt.pth'))
avoid_model.eval()


TRTModule()

In [3]:
camera = CSICamera(width=224, height=224, capture_fps=65)
camera.running = True  # <--- REQUIRED to start frame capture

car = NvidiaRacecar()

image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

detector = Detector(families='tag36h11')

# Control flags
turning = False
turn_start = time.time()
AVOIDANCE_THRESHOLD = 0.7
AVOIDANCE_TURN_ANGLE = -0.8
AVOIDANCE_DURATION = 1.25
STEERING_GAIN = 0.80
STEERING_BIAS = -0.30
NORMAL_THROTTLE = 0.125


Image(value=b'', format='jpeg', height='480', width='640')

In [4]:
def control_loop():
    global turning, turn_start

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Ensure models are on the right device
    road_model.to(device).eval()
    avoid_model.to(device).eval()

    while True:
        frame = camera.value.copy()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        tags = detector.detect(gray)

        # --- AprilTag STOP logic ---
        if tags:
            print(f"[!] AprilTags detected: {[tag.tag_id for tag in tags]}")
            car.throttle = 0.0
            car.steering = 0.0
        else:
            # Preprocess for NN input
            image_input = preprocess(frame).half().to(device)

            # --- Collision Avoidance logic ---
            avoid_output = avoid_model(image_input)
            avoid_prob = float(F.softmax(avoid_output, dim=1).flatten()[0])
            print(f"[DEBUG] Avoidance probability: {avoid_prob:.4f}")

            if avoid_prob < AVOIDANCE_THRESHOLD and not turning:
                # --- Road Following ---
                road_output = road_model(image_input).detach().cpu().numpy().flatten()
                print(f"[DEBUG] Road model output: {road_output}")
                x = float(road_output[0])
                car.steering = x * STEERING_GAIN + STEERING_BIAS
                car.throttle = NORMAL_THROTTLE
            else:
                # --- Obstacle Avoidance ---
                if not turning:
                    print("[INFO] Starting avoidance turn")
                    car.steering = AVOIDANCE_TURN_ANGLE - .05
                    turning = True
                    turn_start = time.time()
                    time.sleep(.45)
                    car.steering = -AVOIDANCE_TURN_ANGLE
                    time.sleep(.8)
                    car.steering = AVOIDANCE_TURN_ANGLE

                if time.time() - turn_start >= AVOIDANCE_DURATION and avoid_prob < AVOIDANCE_THRESHOLD:
                    turning = False

                car.throttle = NORMAL_THROTTLE

        # Draw AprilTags
        for tag in tags:
            for i in range(4):
                cv2.line(frame,
                         tuple(tag.corners[i - 1, :].astype(int)),
                         tuple(tag.corners[i, :].astype(int)),
                         (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {tag.tag_id}",
                        tuple(tag.corners[0, :].astype(int)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        # Show image
        image_widget.value = bgr8_to_jpeg(frame)


In [5]:
thread = threading.Thread(target=control_loop)
thread.start()


In [6]:
car.throttle = 0
car.steering = 0